### Assignment-1

Importing all the required Libraries
```bash
numpy
sklearn
```

In [10]:
import sklearn as sk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hashlib

## Question 1

In [2]:
class HashTable:
    def __init__(self, size):
        self.input = size;
        self.map = [None] * size;


## Question 2
Bloom Filter implementation


In [13]:
class BloomFilter:
    def __init__(self, size, hashfuncCnt):
        self.size = size            ## M : Size of bit array
        self.hashfuncCnt = hashfuncCnt ## K : Number of hash functions 
        self.bitArray = [0] * size

    def addElement(self, element):
        kHash = self.hashFunction(element, seed=42)
        for hash in kHash:
            self.bitArray[hash] = 1

    def existCheck(self, element):
        kHash = self.hashFunction(element, seed=42)
        for hash in kHash:
            if self.bitArray[hash] == 0:
                return False
        return True

    def hashFunction(self, element, seed):
        kHash = []
        for i in range(self.hashfuncCnt):
            hash = (hashlib.md5((str(element) + str(seed) + str(i)).encode()).hexdigest())
            kHash.append(int(hash, 16) % self.size)
        return kHash

In [16]:
bloom = BloomFilter(10000, 5)

# Insert elements
bloom.addElement("85123A")
bloom.addElement("71053")

# Check membership
print(bloom.existCheck("85123A"))  # True (should be)
print(bloom.existCheck("99999"))   # False (most likely)


True
False


In [8]:
retailDf = pd.read_csv('online-retail.csv')
print(retailDf['StockCode'].head())


0    85123A
1     71053
2    84406B
3    84029G
4    84029E
Name: StockCode, dtype: object


### Misa Gries Algorithm 
---
Frequency Estimation in a Data Stream
```
https://www.cs.dartmouth.edu/~ac/Teach/data-streams-lecnotes.pdf
```

In [2]:
class DataStream:
    def __init__(self, thresh: int):
        if thresh <= 0:
            raise ValueError("Threshold must be positive")
        self.thresh = thresh
        self.freq_map = {}

    ## ACTUAL Misra-Gries Algorithm implementation
    def addElement(self, element):
        if element in self.freq_map.keys():
            self.freq_map[element] += 1
        else:
            self.freq_map[element] = 1
        if len(self.freq_map) >= self.thresh:
            keys_to_remove = []
            for key in self.freq_map.keys():
                self.freq_map[key] -= 1
                if self.freq_map[key] == 0:
                    keys_to_remove.append(key)
            for key in keys_to_remove:
                del self.freq_map[key]

    def queryEle(self, element):
        if element in self.freq_map.keys():
            return self.freq_map.get(element,0)
    
    def freqElereturn(self):
        return self.freq_map
        


In [11]:
if __name__== "__main__":
    # patch the method
    dataStream = DataStream(thresh=4000)
    retailDf = pd.read_csv('online-retail.csv')
    stockCodes = retailDf['StockCode']
    
    # Calculate actual frequency of '22913' for debugging
    actual_freq = stockCodes.value_counts().get('22913', 0)
    print(f"Actual frequency of '22913' in dataset: {actual_freq}")
    
    for code in stockCodes:
        dataStream.addElement(code)

    queryCode = '22913'
    result = dataStream.queryEle(queryCode)
    print(f"Approximate frequency of '{queryCode}' from Misra-Gries: {result}")



Actual frequency of '22913' in dataset: 114
Approximate frequency of '22913' from Misra-Gries: 113


```

Direct Frequency count of each element in the Stream

```

In [16]:
dataframe = pd.read_csv('online-retail.csv')
stockCodes = dataframe['StockCode']
direct_freq_map = {}
for code in stockCodes:
    if code in direct_freq_map:
        direct_freq_map[code] += 1
    else:
        direct_freq_map[code] = 1

print("Direct frequency count of elements:")
print(len(direct_freq_map))
# for code, freq in direct_freq_map.items():
#     print(f"Element {code}: {freq}")
    


Direct frequency count of elements:
4070


# Question 3

In [14]:
from sklearn.datasets import fetch_kddcup99
from sklearn.random_projection import GaussianRandomProjection , SparseRandomProjection
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [4]:
def kmeanLoss(data, centroid):
    pass

    

In [5]:
datafd = fetch_kddcup99(subset = None,shuffle=True)
KMEANS = 15
FEATCNT_D = datafd.data.shape[1]
SAMPLECNT_N = datafd.data.shape[0]
print(f"Number of samples: {SAMPLECNT_N}, Number of features: {FEATCNT_D}")


Number of samples: 494021, Number of features: 41


In [13]:
matrix_D = datafd.frame
matrix_D


In [ ]:
X_VARIABLE = [5,20,15,20,25]


In [15]:
data = fetch_kddcup99(subset=None)
D = data.data  # n x d
y = data.target  # n x 1, multi-class strings -> numericize
le = LabelEncoder()
y = le.fit_transform(y).astype(float)  # Treat as numeric for regression
n, d = D.shape
print(f"Dataset shape: n={n}, d={d}")

# ---------------- Question 3(a): K-Means Clustering with JL Projection ----------------
x_values_a = [5, 10, 15, 20, 25]  # Corrected from {5,20,15,20,25} assuming typo
k = 15
n_trials = 5

results_a = {}
for x in x_values_a:
    trial_costs_approx = []
    trial_costs_exact = []
    for trial in range(n_trials):
        # Generate JL matrix M (d x x), Gaussian entries scaled by 1/sqrt(x)
        M = np.random.normal(0, 1.0 / np.sqrt(x), (d, x))
        E = D @ M  # n x x projected data
        
        # K-means on projected E
        kmeans_E = KMeans(n_clusters=k, n_init=1, random_state=trial).fit(E)
        labels = kmeans_E.labels_
        
        # Compute induced clustering cost in original space
        cost_approx = 0.0
        for c in range(k):
            cluster_idx = np.where(labels == c)[0]
            if len(cluster_idx) > 0:
                centroid = np.mean(D[cluster_idx], axis=0)
                cost_approx += np.sum(np.linalg.norm(D[cluster_idx] - centroid, axis=1)**2)
        
        # Exact K-means on original D (inertia_ is the cost)
        kmeans_D = KMeans(n_clusters=k, n_init=1, random_state=trial).fit(D)
        cost_exact = kmeans_D.inertia_
        
        trial_costs_approx.append(cost_approx)
        trial_costs_exact.append(cost_exact)
    
    # Average costs
    avg_approx = np.mean(trial_costs_approx)
    avg_exact = np.mean(trial_costs_exact)
    rel_error = (avg_approx - avg_exact) / avg_exact if avg_exact > 0 else 0
    results_a[x] = {'avg_approx': avg_approx, 'avg_exact': avg_exact, 'rel_error': rel_error}

# Table for (a)
df_a = pd.DataFrame.from_dict(results_a, orient='index')
print("\n3(a) Results: Average Clustering Costs and Relative Error")
print(df_a.round(2))

# Bar graph for relative error
plt.figure(figsize=(8, 5))
plt.bar(df_a.index, df_a['rel_error'], color='skyblue')
plt.xlabel('Projection Dimension x')
plt.ylabel('Relative Error in Cost')
plt.title('Relative Error in K-Means Cost vs. Projection Dim (3a)')
plt.savefig('3a_bar_graph.png')
plt.show()

Dataset shape: n=494021, d=41


TypeError: can't multiply sequence by non-int of type 'float'